# 03 — Evaluation

Compare fine-tuned model vs baseline. Includes automated metrics and LLM-as-judge.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load evaluation results
results_path = Path('../data/evaluation/evaluation_results.json')
if results_path.exists():
    results = json.loads(results_path.read_text())
    print(f'Loaded {len(results)} evaluation results')
else:
    results = []
    print('No evaluation results yet. Run: python scripts/evaluate.py')

In [ ]:
if results:
    # Extract judge scores
    categories = ['tone', 'structure', 'vocabulary', 'authenticity', 'overall']
    
    ft_scores = {cat: [] for cat in categories}
    bl_scores = {cat: [] for cat in categories}
    
    for r in results:
        ft_j = r.get('fine_tuned', {}).get('judge', {})
        bl_j = r.get('baseline', {}).get('judge', {})
        for cat in categories:
            if isinstance(ft_j.get(cat), (int, float)):
                ft_scores[cat].append(ft_j[cat])
            if isinstance(bl_j.get(cat), (int, float)):
                bl_scores[cat].append(bl_j[cat])
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(categories))
    width = 0.35
    
    ft_means = [np.mean(ft_scores[c]) if ft_scores[c] else 0 for c in categories]
    bl_means = [np.mean(bl_scores[c]) if bl_scores[c] else 0 for c in categories]
    
    ax.bar(x - width/2, ft_means, width, label='Fine-tuned', color='#2196F3')
    ax.bar(x + width/2, bl_means, width, label='Baseline', color='#FF9800')
    
    ax.set_ylabel('Score (1-10)')
    ax.set_title('Fine-tuned vs Baseline — LLM Judge Scores')
    ax.set_xticks(x)
    ax.set_xticklabels([c.title() for c in categories])
    ax.legend()
    ax.set_ylim(0, 10)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
if results:
    # Side-by-side comparison of a random example
    import random
    r = random.choice(results)
    
    print('PROMPT:', r['prompt'])
    print()
    print('=== REFERENCE (Jacq\'s actual writing) ===')
    print(r['reference_preview'])
    print()
    print('=== FINE-TUNED ===')
    print(r['fine_tuned']['response_preview'])
    print(f"\nJudge: {r['fine_tuned']['judge']}")
    print()
    print('=== BASELINE ===')
    print(r['baseline']['response_preview'])
    print(f"\nJudge: {r['baseline']['judge']}")